In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch version: 2.10.0+cu128
Device: cuda


In [5]:
MAX_LEN    = 15
EMBED_DIM  = 50
HIDDEN     = 64
LAYERS     = 1
BIDIR      = False
DROPOUT    = 0.3
BATCH_SIZE = 2
EPOCHS     = 10
LR         = 0.001

In [2]:
train_texts = [
    "i love this movie it is amazing",
    "this film was absolutely terrible",
    "wonderful performance loved every minute",
    "worst film i have ever seen",
    "brilliant storyline and great acting",
    "boring and completely predictable",
]
train_labels = [1, 0, 1, 0, 1, 0]

val_texts  = ["really enjoyed this film", "hated every second of it"]
val_labels = [1, 0]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")

Train: 6 | Val: 2


In [3]:
def build_vocab(texts):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for text in texts:
        for word in text.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len):
    words   = text.lower().split()
    indices = [vocab.get(w, 1) for w in words]
    indices = indices + [0] * (max_len - len(indices))
    return indices[:max_len]

vocab = build_vocab(train_texts + val_texts)
print(f"Vocab size: {len(vocab)}")

Vocab size: 35


In [6]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.data   = [torch.tensor(encode(t, vocab, max_len), dtype=torch.long) for t in texts]
        self.labels = [torch.tensor(l, dtype=torch.long) for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

train_dataset = TextDataset(train_texts, train_labels, vocab, MAX_LEN)
val_dataset   = TextDataset(val_texts,   val_labels,   vocab, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print("Datasets ready.")

Datasets ready.


In [7]:
class SentimentLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes,
                 num_layers=1, bidirectional=False, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # ← nn.LSTM instead of nn.RNN — only change in architecture
        self.lstm = nn.LSTM(
            input_size    = embed_dim,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        fc_input = hidden_size * 2 if bidirectional else hidden_size
        self.fc  = nn.Linear(fc_input, num_classes)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        # embedded → (batch, seq_len, embed_dim)

        # ⚡ KEY DIFFERENCE FROM RNN:
        # RNN:  output, hidden          = self.rnn(embedded)
        # LSTM: output, (hidden, cell)  = self.lstm(embedded)
        output, (hidden, cell) = self.lstm(embedded)

        # hidden → (num_layers * directions, batch, hidden_size)
        # cell   → (num_layers * directions, batch, hidden_size)
        # We use hidden[-1], NOT cell[-1]

        last = self.dropout(hidden[-1])
        # last → (batch, hidden_size)

        return self.fc(last)


model = SentimentLSTM(
    vocab_size    = len(vocab),
    embed_dim     = EMBED_DIM,
    hidden_size   = HIDDEN,
    num_classes   = 2,
    num_layers    = LAYERS,
    bidirectional = BIDIR,
    dropout       = DROPOUT
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Notice: LSTM has ~4x more parameters than RNN for same hidden size
# This is because LSTM has 4 weight matrices (f, i, C̃, o gates)

SentimentLSTM(
  (embedding): Embedding(35, 50, padding_idx=0)
  (lstm): LSTM(50, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

Total parameters: 31,576


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Loss & Optimizer ready.")

Loss & Optimizer ready.


In [9]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        predictions = model(X_batch)
        loss        = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (predictions.argmax(1) == y_batch).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss        = criterion(predictions, y_batch)

            total_loss += loss.item()
            correct    += (predictions.argmax(1) == y_batch).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

print("Functions ready.")

Functions ready.


In [10]:
best_val_acc = 0.0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1:02d}/{EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2%}  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2%}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_lstm.pt")
        print(f"  ✓ Best model saved (Val Acc: {best_val_acc:.2%})")

print(f"\nTraining complete. Best Val Accuracy: {best_val_acc:.2%}")

Epoch 01/10  |  Train Loss: 0.6951  Train Acc: 50.00%  |  Val Loss: 0.6932  Val Acc: 50.00%
  ✓ Best model saved (Val Acc: 50.00%)
Epoch 02/10  |  Train Loss: 0.6948  Train Acc: 50.00%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 03/10  |  Train Loss: 0.6878  Train Acc: 66.67%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 04/10  |  Train Loss: 0.6945  Train Acc: 33.33%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 05/10  |  Train Loss: 0.6993  Train Acc: 50.00%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 06/10  |  Train Loss: 0.6885  Train Acc: 66.67%  |  Val Loss: 0.6934  Val Acc: 50.00%
Epoch 07/10  |  Train Loss: 0.6826  Train Acc: 83.33%  |  Val Loss: 0.6935  Val Acc: 50.00%
Epoch 08/10  |  Train Loss: 0.6982  Train Acc: 33.33%  |  Val Loss: 0.6935  Val Acc: 50.00%
Epoch 09/10  |  Train Loss: 0.6912  Train Acc: 50.00%  |  Val Loss: 0.6937  Val Acc: 50.00%
Epoch 10/10  |  Train Loss: 0.6899  Train Acc: 50.00%  |  Val Loss: 0.6939  Val Acc: 50.00%

Training complete. Best Val Accuracy: 50

In [16]:
 def predict(text, model, vocab, max_len):
    model.eval()
    encoded = encode(text, vocab, max_len)
    x       = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(x)
        probs  = torch.softmax(output, dim=1)
        pred   = output.argmax(1).item()

    label      = "Positive 😊" if pred == 1 else "Negative 😞"
    confidence = probs[0][pred].item()
    print(f"Text       : {text}")
    print(f"Prediction : {label}  ({confidence:.2%} confidence)\n")


predict("this movie was wonderful",  model, vocab, MAX_LEN)
predict("i hated this film",   model, vocab, MAX_LEN)
predict("great acting",  model, vocab, MAX_LEN)


Text       : this movie was wonderful
Prediction : Positive 😊  (50.58% confidence)

Text       : i hated this film
Prediction : Positive 😊  (50.53% confidence)

Text       : great acting
Prediction : Positive 😊  (50.61% confidence)

